In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import sys
import numpy as np
import networkx as nx

In [2]:
sys.path.append('../src/nlr_ghe_screen/clustering/')
from clustering import (
    seed_cluster, 
    construct_euclidian_dist_matrix,
    find_seed_centroids,
    grow_cluster
)

sys.path.append('../src/nlr_ghe_screen/api/')
from building_footprint_api import (
    get_overture_footprints,
    get_ornl_footprints
)
from osmnx_api import (
    get_road_network,
    plot_network_graph
)

In [3]:
TEST_BBOX = (47.58, -122.36, 47.61, -122.31) # xmin, ymin, xmax, ymax
south, west, north, east = TEST_BBOX

In [4]:
ghe_locations = gpd.read_file("../src/exports/ghe_locations.geojson")
geom_box = box(west, south, east, north)
cropped_ghes = gpd.clip(ghe_locations, geom_box)

In [5]:
overture_buildings = get_overture_footprints(bbox=TEST_BBOX)
ornl_buildings = get_ornl_footprints(bbox=TEST_BBOX)


---------------------------
Retrieving Overture Maps Data
Success - Data retrieved
---------------------------

---------------------------
Retrieving ORNL Structure Data
total features: 2679
2000/2679
2679/2679
Success - Data retrieved
---------------------------


In [6]:
cropped_ghes = cropped_ghes.to_crs(cropped_ghes.estimate_utm_crs())
ornl_buildings = ornl_buildings.to_crs(cropped_ghes.crs)
overture_buildings = overture_buildings.to_crs(cropped_ghes.crs)

In [9]:
network = get_road_network(bbox=TEST_BBOX)


---------------------------
Post - Networkx road network request
Success - Query Received
---------------------------


In [11]:
m = plot_network_graph(network, overture_buildings)
m

In [ ]:
buildings_joined = gpd.sjoin(
    ornl_buildings,
    overture_buildings,
    how='left',
    predicate='intersects'
).drop(columns='index_right')

# Set all primary occupancy fields to apartment 
# if the data from Overture Maps says so
target_vals = ['apartments']
mask = buildings_joined['class'].isin(target_vals)
buildings_joined.loc[mask, 'PRIM_OCC'] = buildings_joined.loc[mask, 'class']

# Use the building height from overture maps in the case where
# ORNL Structures doesn't have any data
mask = buildings_joined['HEIGHT'].isna()
buildings_joined.loc[mask, 'HEIGHT'] = buildings_joined.loc[mask, 'height']


In [ ]:
dist_matrix = construct_euclidian_dist_matrix(
    buildings_df=cropped_ghes,
    max_distance=250
)

labels = seed_cluster(
    dist_matrix=dist_matrix,
    max_distance=250
)
cropped_ghes['cluster'] = labels

In [ ]:
centroids = find_seed_centroids(
    ghe_locations=cropped_ghes,
    cluster_col='cluster'
)

In [ ]:
clusters = grow_cluster(
    seed_centroids=centroids,
    building_data=buildings_joined,
)

In [ ]:
m = plot_network_graph(network, clusters)
m

In [ ]:
c1 = cropped_ghes['cluster'].value_counts().rename("n1")
c2 = clusters['cluster'].value_counts().rename("n2")

counts = pd.concat([c1, c2], axis=1).fillna(0).astype(int)
counts["min"] = counts[["n1", "n2"]].min(axis=1)

ranked = counts.sort_values("min", ascending=False)      
ordered_labels = ranked.index.tolist()
ordered_labels[:10]

In [ ]:
idx = 0
m = centroids[centroids['cluster'] == idx].explore(color='red')
cropped_ghes[cropped_ghes['cluster'] == idx].explore(m=m, color='green')
clusters[clusters['cluster'] == idx].explore(m=m, color='blue')